<a href="https://colab.research.google.com/github/Strangeabhi/ML-II-Abhishek_C_Patil-4NI23CI005/blob/main/Foil_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Foil Algorithm

In [2]:
import pandas as pd

df = pd.read_csv("chess.csv")

print("Successfully accessed 'chess.csv'. Here's a preview of the data:")
print(df.head())

Successfully accessed 'chess.csv'. Here's a preview of the data:
   a  1  b  3  c  2  draw
0  a  1  c  1  c  2  draw
1  a  1  c  1  d  1  draw
2  a  1  c  1  d  2  draw
3  a  1  c  2  c  1  draw
4  a  1  c  2  c  3  draw

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28055 entries, 0 to 28054
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   a       28055 non-null  object
 1   1       28055 non-null  int64 
 2   b       28055 non-null  object
 3   3       28055 non-null  int64 
 4   c       28055 non-null  object
 5   2       28055 non-null  int64 
 6   draw    28055 non-null  object
dtypes: int64(3), object(4)
memory usage: 1.5+ MB


In [3]:
import math

def cover(rule, data_list_of_dicts):
    return [x for x in data_list_of_dicts if all(x.get(k) == v for k, v in rule)]

def foil(data_list_of_dicts, target_attribute):
    initial_positive_examples = [x for x in data_list_of_dicts if x[target_attribute] == "Yes"]
    initial_negative_examples = [x for x in data_list_of_dicts if x[target_attribute] == "No"]

    learned_rules = []
    current_uncovered_positives = list(initial_positive_examples)

    while current_uncovered_positives:
        current_rule = []

        while cover(current_rule, initial_negative_examples):
            best_literal = None
            max_gain = -float('inf')

            if not data_list_of_dicts:
                break

            all_attributes = list(data_list_of_dicts[0].keys())

            for attribute_key in all_attributes:
                if attribute_key == target_attribute:
                    continue

                unique_values_for_attribute = set(instance[attribute_key] for instance in data_list_of_dicts)

                for attribute_value in unique_values_for_attribute:
                    candidate_literal = (attribute_key, attribute_value)

                    if candidate_literal in current_rule:
                        continue

                    candidate_extended_rule = current_rule + [candidate_literal]

                    P0 = len(cover(current_rule, current_uncovered_positives))
                    N0 = len(cover(current_rule, initial_negative_examples))

                    P1 = len(cover(candidate_extended_rule, current_uncovered_positives))
                    N1 = len(cover(candidate_extended_rule, initial_negative_examples))

                    if P1 > 0 and (P1 + N1) > 0:
                        if P0 > 0 and (P0 + N0) > 0:
                             gain = P1 * (
                                math.log2(P1 / (P1 + N1)) -
                                math.log2(P0 / (P0 + N0))
                            )
                        else:
                            gain = P1 * math.log2(P1 / (P1 + N1))

                        if gain > max_gain:
                            max_gain = gain
                            best_literal = candidate_literal

            if best_literal is None:
                break

            current_rule.append(best_literal)

        covered_by_this_rule = cover(current_rule, current_uncovered_positives)
        if covered_by_this_rule:
            learned_rules.append(current_rule)
            current_uncovered_positives = [x for x in current_uncovered_positives if x not in covered_by_this_rule]
        else:
            break

    return learned_rules


column_names = [
    'white_king_file', 'white_king_rank',
    'white_rook_file', 'white_rook_rank',
    'black_king_file', 'black_king_rank',
    'optimal_depth_of_win'
]

df_chess = pd.read_csv('chess.csv', names=column_names)

target_col_name = 'optimal_depth_of_win'
positive_class_value = 'zero'

foil_target_attribute_name = 'is_win_in_zero_moves'
df_chess[foil_target_attribute_name] = df_chess[target_col_name].apply(
    lambda x: 'Yes' if str(x).strip().lower() == positive_class_value else 'No'
)

feature_columns = [col for col in column_names if col != target_col_name]
for col in feature_columns:
    df_chess[col] = df_chess[col].astype(str)

foil_data_columns = feature_columns + [foil_target_attribute_name]
foil_dataset = df_chess[foil_data_columns].to_dict(orient='records')

if not foil_dataset:
    print("Error: No data available after preprocessing for FOIL algorithm. Check your data and preprocessing steps.")
else:
    print(f"\nApplying FOIL algorithm to find rules where '{foil_target_attribute_name}' is 'Yes' (i.e., original '{target_col_name}' is '{positive_class_value}')...")

    generated_rules = foil(foil_dataset, foil_target_attribute_name)

    print("\n--- Generated Rules for winning in zero moves ---")
    if not generated_rules:
        print(f"No rules found for target outcome '{positive_class_value}'.")
        print("This might happen if the target is not frequent or highly separable by simple rules or if the dataset is too small.")
    else:
        for i, rule in enumerate(generated_rules):
            rule_antecedent = " AND ".join(f"{k}={v}" for k, v in rule)
            print(f"Rule {i+1}: IF {rule_antecedent} THEN {foil_target_attribute_name}='Yes' (original '{target_col_name}' is '{positive_class_value}')")


Applying FOIL algorithm to find rules where 'is_win_in_zero_moves' is 'Yes' (i.e., original 'optimal_depth_of_win' is 'zero')...

--- Generated Rules for winning in zero moves ---
Rule 1: IF black_king_rank=1 AND white_rook_file=a AND black_king_file=a AND white_king_file=c AND white_rook_rank=6 THEN is_win_in_zero_moves='Yes' (original 'optimal_depth_of_win' is 'zero')
Rule 2: IF black_king_rank=1 AND white_rook_file=a AND black_king_file=a AND white_king_file=c AND white_rook_rank=7 THEN is_win_in_zero_moves='Yes' (original 'optimal_depth_of_win' is 'zero')
Rule 3: IF black_king_rank=1 AND white_rook_rank=1 AND white_king_rank=3 AND black_king_file=d AND white_king_file=d AND white_rook_file=b THEN is_win_in_zero_moves='Yes' (original 'optimal_depth_of_win' is 'zero')
Rule 4: IF black_king_rank=1 AND white_rook_file=a AND black_king_file=a AND white_king_file=c AND white_rook_rank=5 THEN is_win_in_zero_moves='Yes' (original 'optimal_depth_of_win' is 'zero')
Rule 5: IF black_king_ran

In [4]:
print(generated_rules)

[[('black_king_rank', '1'), ('white_rook_file', 'a'), ('black_king_file', 'a'), ('white_king_file', 'c'), ('white_rook_rank', '6')], [('black_king_rank', '1'), ('white_rook_file', 'a'), ('black_king_file', 'a'), ('white_king_file', 'c'), ('white_rook_rank', '7')], [('black_king_rank', '1'), ('white_rook_rank', '1'), ('white_king_rank', '3'), ('black_king_file', 'd'), ('white_king_file', 'd'), ('white_rook_file', 'b')], [('black_king_rank', '1'), ('white_rook_file', 'a'), ('black_king_file', 'a'), ('white_king_file', 'c'), ('white_rook_rank', '5')], [('black_king_rank', '1'), ('white_rook_rank', '1'), ('white_king_rank', '3'), ('black_king_file', 'c'), ('white_king_file', 'c'), ('white_rook_file', 'g')], [('black_king_rank', '1'), ('white_rook_rank', '1'), ('white_king_rank', '3'), ('black_king_file', 'd'), ('white_king_file', 'd'), ('white_rook_file', 'g')], [('white_rook_file', 'a'), ('black_king_file', 'a'), ('white_king_file', 'c'), ('black_king_rank', '1'), ('white_rook_rank', '8')